# SSAlign Structure Search

**SSAlign** is an ultrafast, sensitive protein structure search method. It runs a two-stage retrieve-then-refine pipeline: each structure is reduced to a Foldseek 3Di sequence, embedded with **SaProt-650M** (a structure-aware protein language model), and a **whitened FAISS cosine prefilter** retrieves the top candidates per query. In refine mode (`mode=1`), borderline candidates whose cosine falls below a threshold are re-ranked by **SAligner**, a 3Di global aligner, recovering sensitivity that pure embedding cosine would miss.

The `ssalign-search` tool exposes this method with two target sources: **mode 1** builds a FAISS index on the fly from a supplied set of target structures, and **mode 2** searches a prebuilt SSAlignDB directory for repeated large-scale search.

References:
- Wang et al., *bioRxiv* 2025 ([preprint](https://www.biorxiv.org/content/10.1101/2025.07.03.662911v1)) — SSAlign
- [ISYSLAB-HUST/SSAlign](https://github.com/ISYSLAB-HUST/SSAlign) — reference implementation and web server

> **Note:** SaProt-650M (~2.5 GB) is downloaded from HuggingFace on first run, and a GPU is recommended for embedding throughput. The standalone environment is built automatically the first time the tool is called.

In [1]:
from proto_tools.tools.structure_alignment.ssalign import (
    run_ssalign_search,
    SSAlignSearchInput,
    SSAlignSearchConfig,
    SSAlignQuery,
)
from proto_tools.entities import Structure
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Input fields for the search.
display_api_reference("ssalign-search", "input", "run_ssalign_search")

**Input** — `SSAlignSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `queries` | `list[proto_tools.tools.structure_alignment.ssalign.ssalign_search.SSAlignQuery]` | required | Query structures to search; one ranked hit-bundle is returned per query, in order |

In [3]:
# Config fields: target source (mode 1 vs mode 2), search mode, prefilter and refine options.
display_api_reference("ssalign-search", "config", "run_ssalign_search")

**Config** — `SSAlignSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device for SaProt embedding, e.g. 'cuda' (NVIDIA GPU) or 'cpu' |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `target_structures` | `list[proto_tools.entities.structures.structure.Structure] \| None` | `None` | Mode 1: targets to embed and index in-memory. Provide this OR ssalign_db, not both |
| `ssalign_db` | `str \| None` | `None` | Mode 2: path or AssetRef to a prebuilt SSAlignDB dir. Provide this OR target_structures |
| `mode` | `Literal[0, 1]` | `1` | 0 = cosine prefilter only; 1 = prefilter + SAligner 3Di refine of below-threshold hits |
| `dim` | `Literal[64, 128, 256, 512, 1280]` | `512` | Prebuilt-SSAlignDB index to search (mode 2); ignored when building on the fly |
| `prefilter_target` | `int` | `2000` | Top-K cosine candidates kept from the FAISS prefilter per query |
| `prefilter_threshold` | `float` | `0.3` | Cosine cutoff; below-threshold candidates are SAligner-refined in mode 1 |
| `max_target` | `int` | `1000` | Final hits returned per query after ranking; must be <= prefilter_target |
| `batch_size` | `int` | `8` | Structures per SaProt forward pass |
| `num_threads` | `int` | `4` | CPU threads for foldseek 3Di extraction and the FAISS search |

In [4]:
# Output: one ranked hit-bundle per query.
display_api_reference("ssalign-search", "output", "run_ssalign_search")

**Output** — `SSAlignSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.structure_alignment.ssalign.ssalign_search.SSAlignQueryResult]` | required | One ranked hit-bundle per query, in input order |

## Mode 1: build a FAISS index on the fly from a set of target structures

In mode 1 you supply the target structures directly via `config.target_structures`. SSAlign embeds and indexes them in memory at call time, then searches each query against that index. This is the right choice for ad-hoc searches against a modest target set.

In [5]:
# Load a few real structures from the repo's bundled test fixtures as the target set,
# and reuse one of them (PD-L1) as the query so we expect a strong self-hit. The data
# dir is resolved by package location so this runs from any working directory.
# NOTE: the first call builds the standalone env automatically and downloads
# SaProt-650M (~2.5 GB); a GPU is recommended but device="cpu" works.
from pathlib import Path

import proto_tools

data_dir = Path(proto_tools.__file__).resolve().parent.parent / "tests" / "dummy_data"
targets = [
    Structure.from_file(data_dir / "pdl1.pdb"),
    Structure.from_file(data_dir / "renin_af3.pdb"),
    Structure.from_file(data_dir / "test_structure_similarity.pdb"),
]
query_structure = Structure.from_file(data_dir / "pdl1.pdb")

result = run_ssalign_search(
    SSAlignSearchInput(queries=[SSAlignQuery(structure=query_structure, query_id="q1")]),
    SSAlignSearchConfig(
        target_structures=targets,
        device="cpu",
        mode=1,
        max_target=5,
        prefilter_target=5,
    ),
)

# Inspect the top hits for the first query.
for hit in result.results[0].hits[:5]:
    print(
        f"rank={hit.rank}  target_id={hit.target_id:<10}  "
        f"prefilter_score={hit.prefilter_score:.3f}  ss_score={hit.ss_score:.3f}"
    )

Running run_ssalign_search [00:00]

rank=1  target_id=target_0    prefilter_score=1.000  ss_score=1.110
rank=2  target_id=target_1    prefilter_score=0.775  ss_score=0.986
rank=3  target_id=target_2    prefilter_score=0.631  ss_score=0.907


## Mode 2: search a prebuilt SSAlignDB directory

For repeated large-scale search, point `config.ssalign_db` at a prebuilt SSAlignDB directory (FAISS index + whitening mu/W + id/seq npz) instead of supplying `target_structures`. The two sources are mutually exclusive — provide exactly one. Mode 2 always applies the database's shipped whitening, and `dim` must match the index dimension shipped in that directory.

In [6]:
# Illustrative: mode 2 searches a manually downloaded SSAlignDB directory.
# Prebuilt databases (SwissProt, AFDB, ...) are available from the SSAlign web
# server: http://bioinfo.isyslab.info/ssalign/  . When the directory is absent or
# incomplete, run_ssalign_search raises MissingAssetError before any compute, so we
# catch it here to keep the demo runnable.
from proto_tools.utils.tool_io import MissingAssetError

try:
    db_result = run_ssalign_search(
        SSAlignSearchInput(queries=[SSAlignQuery(structure=query_structure, query_id="q1")]),
        SSAlignSearchConfig(
            ssalign_db="/path/to/SSAlignDB/SwissProt",
            dim=512,
            device="cpu",
            max_target=5,
            prefilter_target=5,
        ),
    )
    print(f"{db_result.results[0].num_hits} hits for query {db_result.results[0].query_id}")
except MissingAssetError as e:
    print(f"SSAlignDB not provisioned (expected in this demo): {e}")

ERROR: Tool ssalign-search: failed with MissingAssetError: ssalign: database not provisioned
SSAlignDB directory '/path/to/SSAlignDB/SwissProt' is missing or lacks required files for dim=512 (expected *_IndexFlatIP_512_faiss.index, *_whitening_mu.npy, *_whitening_W.npy, *_id_Seq.npz).


SSAlignDB not provisioned (expected in this demo): ssalign: database not provisioned
SSAlignDB directory '/path/to/SSAlignDB/SwissProt' is missing or lacks required files for dim=512 (expected *_IndexFlatIP_512_faiss.index, *_whitening_mu.npy, *_whitening_W.npy, *_id_Seq.npz).


In [7]:
# Export the mode-1 results to the example_output/ directory (JSON is the default;
# "csv" is also supported — one row per hit with query_id, rank, target_id, scores, and refined).
result.export("ssalign_demo", export_path="example_output", file_format="json")